# CMPE 58T — Turkish Dependency Parsing
## 03 — Ablation Study (Grünewald et al. 2021 STEPS methodology)

**Three binary comparisons on BOUN:**
- A1: BERT encoder, no BiLSTM (A1a) vs BiLSTM + BERTurk (A1b)
- A2: BiLSTM + BERTurk, no POS (A2a) vs with POS (A2b)
- A3: BERTurk encoder (A3a) vs XLM-R encoder (A3b)

**Key design decisions (carried from 02_baseline):**
- `run_inference` uses `parser.predict()` — not the manual batch loop
- UAS/LAS from `parser.evaluate()` — authoritative, avoids tensor-indexing bugs
- `parser.args.update(SUPAR_EXTRA)` after every `.load()` call
- `force_rerun=True` in eval cell

**Checkpoint reuse:**
- A1a = A3a = `baseline_boun.pt` (from 02_baseline, no retraining)
- A2a = A1b (same model, different label)

---
## 0 — Mount Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os, glob

PROJECT_DIR  = "/content/drive/MyDrive/58T - Application Project"
DATA_DIR     = os.path.join(PROJECT_DIR, "data")
VOCAB_DIR    = os.path.join(PROJECT_DIR, "vocab")
MODEL_DIR    = os.path.join(PROJECT_DIR, "models")
ABLATION_DIR = os.path.join(MODEL_DIR,   "ablation")
PRED_DIR     = os.path.join(PROJECT_DIR, "predictions")
RESULTS_DIR  = os.path.join(PROJECT_DIR, "results")

for d in [DATA_DIR, VOCAB_DIR, MODEL_DIR, ABLATION_DIR, PRED_DIR, RESULTS_DIR]:
    os.makedirs(d, exist_ok=True)

def get_splits(tb_name):
    folder = os.path.join(DATA_DIR, f"UD_Turkish-{tb_name}")
    files  = sorted(glob.glob(os.path.join(folder, "*.conllu")))
    splits = {}
    for f in files:
        name = os.path.basename(f)
        if   "train" in name: splits["train"] = f
        elif "dev"   in name: splits["dev"]   = f
        elif "test"  in name: splits["test"]  = f
    return splits

BOUN = get_splits("BOUN")
IMST = get_splits("IMST")
PUD  = get_splits("PUD")

# Checkpoint paths
BASELINE_BOUN = os.path.join(MODEL_DIR,    "baseline_boun.pt")  # from 02_baseline
A1B_PATH      = os.path.join(ABLATION_DIR, "a1b_lstm_bert_feat.pt")
A2B_PATH      = os.path.join(ABLATION_DIR, "a2b_lstm_with_pos.pt")
A3B_PATH      = os.path.join(ABLATION_DIR, "a3b_xlmr.pt")

if not os.path.exists(BASELINE_BOUN):
    raise FileNotFoundError("baseline_boun.pt not found. Run 02_baseline.ipynb first.")
print(f"baseline_boun.pt : OK ({os.path.getsize(BASELINE_BOUN):,} B)")
print()
for name, path in [("A1b", A1B_PATH), ("A2b", A2B_PATH), ("A3b XLM-R", A3B_PATH)]:
    status = f"exists ({os.path.getsize(path):,} B)" if os.path.exists(path) else "new — will train"
    print(f"  {name:<12} {status}")


Mounted at /content/drive
baseline_boun.pt : OK (457,210,255 B)

  A1b          new — will train
  A2b          new — will train
  A3b XLM-R    new — will train


---
## 1 — Install & Patch

In [2]:
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118 -q
!pip install "transformers==4.35.2" -q
!pip install supar -q
!pip install conllu -q
!pip install scipy -q

import os
PARSER_PY = "/usr/local/lib/python3.12/dist-packages/supar/parsers/parser.py"
OLD = 'state = torch.load(path if os.path.exists(path) else download(supar.MODEL[src].get(path, path), reload=reload))'
NEW = 'state = torch.load(path if os.path.exists(path) else download(supar.MODEL[src].get(path, path), reload=reload), weights_only=False)'
with open(PARSER_PY) as f: src = f.read()
if OLD in src:
    with open(PARSER_PY, 'w') as f: f.write(src.replace(OLD, NEW))
    print("Patch applied: torch.load weights_only=False")
else:
    print("Patch already applied")

from transformers import AdamW
import torch, transformers
print(f"transformers : {transformers.__version__}")
print(f"PyTorch      : {torch.__version__}")
print(f"GPU          : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE — switch runtime!'}")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 123.5/123.5 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.9/7.9 MB 81.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 49.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 112.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 5.4.1 requires transformers<6.0.0,>=4.41.0, but you have transformers 4.35.2 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 93.1/93.1 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 34.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 773.7/773.7 kB 56.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 51.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 418.7/418.7 kB 43.0 MB/s 

/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(


transformers : 4.35.2
PyTorch      : 2.10.0+cu128
GPU          : NVIDIA A100-SXM4-40GB


---
## 2 — Hyperparameters

In [3]:
BERTURK_NAME = "dbmdz/bert-base-turkish-uncased"
XLMR_NAME    = "xlm-roberta-base"
HIDDEN_SIZE  = 800
DROPOUT      = 0.33
SEED         = 42

# ── BERT-encoder mode (A1a, A3a, A3b BERTurk) ────────────────────────────────
BERT_BATCH      = 5000
BERT_MAX_EPOCHS = 50
BERT_PATIENCE   = 15
BERT_LR         = 1e-3    # task head LR — same as baseline
BERT_ENCODER_LR = 2e-5    # BERTurk encoder LR
BERT_WARMUP     = 0.1
BERT_WD         = 0.01
BERT_EPS        = 1e-8

# ── XLM-R specific (A3b) ─────────────────────────────────────────────────────
XLMR_BATCH      = 2000    # reduced VRAM footprint for XLM-R (280M params)
XLMR_MAX_EPOCHS = 50
XLMR_PATIENCE   = 15
XLMR_LR         = 5e-4    # lower task LR for XLM-R
XLMR_ENCODER_LR = 5e-6    # 4x lower than BERTurk — XLM-R needs gentler fine-tuning

# ── LSTM-encoder mode (A1b, A2a, A2b) ────────────────────────────────────────
LSTM_BATCH       = 5000
LSTM_MAX_EPOCHS  = 50
LSTM_PATIENCE    = 15
LSTM_LR          = 2e-3
LSTM_MU          = 0.9
LSTM_NU          = 0.9
LSTM_EPS         = 1e-12
LSTM_WD          = 0.0
LSTM_DECAY       = 0.75
LSTM_DECAY_STEPS = 5000

# Must be applied via parser.args.update() after every .load()
SUPAR_EXTRA = dict(
    mbr=True, partial=False, tree=False,
    proj=False, punct=False, update_steps=1, clip=5.0
)

print(f"BERT-mode  : LR={BERT_LR}  ENCODER_LR={BERT_ENCODER_LR}  EPOCHS={BERT_MAX_EPOCHS}")
print(f"LSTM-mode  : LR={LSTM_LR}  EPOCHS={LSTM_MAX_EPOCHS}")
print(f"XLM-R      : LR={XLMR_LR}  ENCODER_LR={XLMR_ENCODER_LR}  BATCH={XLMR_BATCH}  EPOCHS={XLMR_MAX_EPOCHS}")


BERT-mode  : LR=0.001  ENCODER_LR=2e-05  EPOCHS=50
LSTM-mode  : LR=0.002  EPOCHS=50
XLM-R      : LR=0.0005  ENCODER_LR=5e-06  BATCH=2000  EPOCHS=50


---
## 3 — Helpers

Same pattern as 02_baseline: `parser.predict()` for CoNLL-U, `parser.evaluate()` for UAS/LAS, `score_predictions()` only for per-sentence npy arrays.

In [4]:
import torch
import numpy as np
import conllu as conllu_lib
import json
from collections import defaultdict
from supar import CRF2oDependencyParser

def load_gold(conllu_path):
    with open(conllu_path, encoding="utf-8") as f:
        return conllu_lib.parse(f.read())

def run_inference(parser, conllu_path, pred_conllu_path, force_rerun=False):
    """
    Write predicted CoNLL-U using parser.predict() — guaranteed correct output.
    Uses cached file if force_rerun=False and file already exists.
    """
    if (not force_rerun
            and os.path.exists(pred_conllu_path)
            and os.path.getsize(pred_conllu_path) > 0):
        print(f"    Cached: {os.path.basename(pred_conllu_path)}")
        with open(pred_conllu_path, encoding="utf-8") as f:
            return conllu_lib.parse(f.read())

    parser.predict(conllu_path, pred=pred_conllu_path, verbose=False)

    with open(pred_conllu_path, encoding="utf-8") as f:
        pred_sents = conllu_lib.parse(f.read())

    print(f"    Saved: {os.path.basename(pred_conllu_path)} ({len(pred_sents)} sentences)")
    return pred_sents

def score_predictions(gold_sents, pred_sents):
    """Sentence-level boolean arrays for McNemar's tests only."""
    total = correct_arc = correct_both = 0
    sent_uas, sent_las = [], []
    for gold, pred in zip(gold_sents, pred_sents):
        gt = [t for t in gold if isinstance(t["id"], int)]
        pt = [t for t in pred if isinstance(t["id"], int)]
        n = len(gt)
        if n == 0: continue
        arc_ok  = sum(1 for g, p in zip(gt, pt) if g["head"] == p["head"])
        both_ok = sum(1 for g, p in zip(gt, pt)
                      if g["head"] == p["head"] and g["deprel"] == p["deprel"])
        total += n; correct_arc += arc_ok; correct_both += both_ok
        sent_uas.append(arc_ok == n); sent_las.append(both_ok == n)
    uas = round(100 * correct_arc  / total, 2) if total else 0.0
    las = round(100 * correct_both / total, 2) if total else 0.0
    return uas, las, np.array(sent_uas, dtype=bool), np.array(sent_las, dtype=bool)

def per_relation_las(gold_sents, pred_sents):
    counts = defaultdict(lambda: [0, 0])
    for gold, pred in zip(gold_sents, pred_sents):
        gt = [t for t in gold if isinstance(t["id"], int)]
        pt = [t for t in pred if isinstance(t["id"], int)]
        for g, p in zip(gt, pt):
            counts[g["deprel"]][1] += 1
            if g["head"] == p["head"] and g["deprel"] == p["deprel"]:
                counts[g["deprel"]][0] += 1
    return {rel: {"correct": c, "total": t,
                  "LAS": round(100 * c / t, 2) if t else 0.0}
            for rel, (c, t) in sorted(counts.items(), key=lambda x: -x[1][1])}

def evaluate_and_save(parser, system_name, conllu_path, test_name,
                      force_rerun=False):
    """
    Run inference with parser.predict(), score with parser.evaluate().
    Saves CoNLL-U and per-sentence npy for McNemar's tests.
    UAS/LAS come from parser.evaluate() — not the manual scorer.
    """
    safe        = system_name.replace(" ", "_")
    pred_conllu = os.path.join(PRED_DIR, f"{safe}_{test_name}.conllu")
    pred_npy    = os.path.join(PRED_DIR, f"{safe}_{test_name}_sent_las.npy")

    print(f"  [{system_name}] → {test_name}")
    pred_sents = run_inference(parser, conllu_path, pred_conllu, force_rerun=force_rerun)

    # Authoritative UAS/LAS from SuPar's own evaluator
    parser.args.update(SUPAR_EXTRA)
    _, metric = parser.evaluate(conllu_path, verbose=False)
    uas = round(metric.uas * 100, 2)
    las = round(metric.las * 100, 2)

    # Sentence-level arrays for McNemar's — load gold fresh
    gold_fresh = load_gold(conllu_path)
    _, _, s_uas, s_las = score_predictions(gold_fresh, pred_sents)
    np.save(pred_npy, s_las)

    print(f"    UAS {uas:.2f}  LAS {las:.2f}  npy: {os.path.basename(pred_npy)}")
    return {"system": system_name, "test": test_name, "UAS": uas, "LAS": las,
            "pred_conllu": os.path.basename(pred_conllu),
            "pred_npy":    os.path.basename(pred_npy)}

print("Helpers defined.")


Helpers defined.


---
## 4 — Training Functions

In [5]:
import supar
from supar import CRF2oDependencyParser
print(f"SuPar: {supar.__version__}")

def train_bert_mode(run_name, encoder_name, lr=BERT_LR, encoder_lr=BERT_ENCODER_LR,
                    max_epochs=BERT_MAX_EPOCHS, patience=BERT_PATIENCE,
                    batch_size=BERT_BATCH):
    """Train in BERT-encoder mode (no BiLSTM). Used for A3b (XLM-R)."""
    model_path = os.path.join(ABLATION_DIR, f"{run_name}.pt")
    print(f"\n{'='*60}")
    print(f" {run_name} | encoder={encoder_name}")
    print(f" LR={lr}  ENCODER_LR={encoder_lr}  EPOCHS={max_epochs}  PATIENCE={patience}")
    print(f"{'='*60}")

    if os.path.exists(model_path):
        print("Checkpoint found — skipping training.")
        parser = CRF2oDependencyParser.load(model_path)
        parser.args.update(SUPAR_EXTRA)
        return parser

    parser = CRF2oDependencyParser.build(
        path=model_path, build=True, encoder="bert",
        bert=encoder_name, train=BOUN["train"],
    )
    parser.train(
        train=BOUN["train"], dev=BOUN["dev"], test=BOUN["test"],
        batch_size=batch_size, epochs=max_epochs, patience=patience,
        lr=lr, lr_rate=encoder_lr, warmup=BERT_WARMUP,
        weight_decay=BERT_WD, eps=BERT_EPS,
        verbose=True, checkpoint=False, seed=SEED, device="0",
    )
    parser.args.update(SUPAR_EXTRA)
    return parser

def train_lstm_mode(run_name, feat):
    """
    Train in LSTM-encoder mode (BiLSTM on top of BERTurk).
    feat=["bert"]        → A1b / A2a  (no POS)
    feat=["bert","tag"]  → A2b        (with POS)
    """
    model_path = os.path.join(ABLATION_DIR, f"{run_name}.pt")
    print(f"\n{'='*60}")
    print(f" {run_name} | encoder=lstm  feat={feat}")
    print(f" LR={LSTM_LR}  EPOCHS={LSTM_MAX_EPOCHS}  PATIENCE={LSTM_PATIENCE}")
    print(f"{'='*60}")

    if os.path.exists(model_path):
        print("Checkpoint found — skipping training.")
        parser = CRF2oDependencyParser.load(model_path)
        parser.args.update(SUPAR_EXTRA)
        return parser

    parser = CRF2oDependencyParser.build(
        path=model_path, build=True, encoder="lstm",
        bert=BERTURK_NAME, train=BOUN["train"],
        feat=feat,
    )
    parser.train(
        train=BOUN["train"], dev=BOUN["dev"], test=BOUN["test"],
        batch_size=LSTM_BATCH, epochs=LSTM_MAX_EPOCHS, patience=LSTM_PATIENCE,
        lr=LSTM_LR, mu=LSTM_MU, nu=LSTM_NU, eps=LSTM_EPS,
        weight_decay=LSTM_WD, decay=LSTM_DECAY, decay_steps=LSTM_DECAY_STEPS,
        verbose=True, checkpoint=False, seed=SEED, device="0",
    )
    parser.args.update(SUPAR_EXTRA)
    return parser

print("train_bert_mode() and train_lstm_mode() ready.")


SuPar: 1.1.4
train_bert_mode() and train_lstm_mode() ready.


---
## 5 — A1a / A3a: BERTurk encoder, no BiLSTM

Reuses `baseline_boun.pt` — no training needed.

In [6]:
# A1a: BERTurk encoder, no BiLSTM — reuses baseline_boun.pt (no training needed)
print("A1a / A3a: Loading baseline_boun.pt (reuses 02_baseline checkpoint)")
parser_a1a = CRF2oDependencyParser.load(BASELINE_BOUN)
parser_a1a.args.update(SUPAR_EXTRA)
parser_a3a = parser_a1a   # A3a is identical to A1a
print("  Loaded. parser_a3a = parser_a1a")


A1a / A3a: Loading baseline_boun.pt (reuses 02_baseline checkpoint)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/59.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


pytorch_model.bin:   0%|          | 0.00/445M [00:00<?, ?B/s]

  Loaded. parser_a3a = parser_a1a


---
## 6 — A1b: BiLSTM + BERTurk features

Expected runtime: ~2–4 hours on A100 (50 epochs max, early stop ~20–30).

In [8]:
def train_lstm_mode(run_name, feat):
    model_path = os.path.join(ABLATION_DIR, f"{run_name}.pt")
    print(f"\n{'='*60}")
    print(f" {run_name} | encoder=lstm  feat={feat}")
    print(f" LR={LSTM_LR}  EPOCHS={LSTM_MAX_EPOCHS}  PATIENCE={LSTM_PATIENCE}")
    print(f"{'='*60}")

    if os.path.exists(model_path):
        print("Checkpoint found — skipping training.")
        parser = CRF2oDependencyParser.load(model_path)
        parser.args.update(SUPAR_EXTRA)
        return parser

    parser = CRF2oDependencyParser.build(
        path=model_path, build=True, encoder="lstm",
        bert=BERTURK_NAME, train=BOUN["train"],
        feat=feat,
        embed=None,   # no pretrained word embeddings — required by LSTM build path
        unk="unk",    # unknown token for word vocab — required by LSTM build path
    )
    parser.train(
        train=BOUN["train"], dev=BOUN["dev"], test=BOUN["test"],
        batch_size=LSTM_BATCH, epochs=LSTM_MAX_EPOCHS, patience=LSTM_PATIENCE,
        lr=LSTM_LR, mu=LSTM_MU, nu=LSTM_NU, eps=LSTM_EPS,
        weight_decay=LSTM_WD, decay=LSTM_DECAY, decay_steps=LSTM_DECAY_STEPS,
        verbose=True, checkpoint=False, seed=SEED, device="0",
    )
    parser.args.update(SUPAR_EXTRA)
    return parser

print("train_lstm_mode() fixed.")

train_lstm_mode() fixed.


In [9]:
# A1b: BiLSTM + BERTurk features (no POS)
# Expected runtime: ~2-4 hours on A100 (50 epochs, early stop ~20-30)
parser_a1b = train_lstm_mode("a1b_lstm_bert_feat", feat=["bert"])



 a1b_lstm_bert_feat | encoder=lstm  feat=['bert']
 LR=0.002  EPOCHS=50  PATIENCE=15


INFO:supar:Loading the data
  0%|                  | 0/7803 00:00<?, ?it/s/usr/local/lib/python3.12/dist-packages/supar/utils/fn.py:155: UserWarning: Using a non-tuple sequence for multidimensional indexing is deprecated and will be changed in pytorch 2.9; use x[tuple(seq)] instead of x[seq]. In pytorch 2.9 this will be interpreted as tensor index, x[torch.tensor(seq)], which will result either in an error or a different result (Triggered internally at /pytorch/torch/csrc/autograd/python_variable_indexing.cpp:347.)
  out_tensor[i][[slice(-i, None) if padding_side == 'left' else slice(0, i) for i in tensor.size()]] = tensor
INFO:supar:
train: Dataset(n_sentences=7803, n_batches=41, n_buckets=32)
dev:   Dataset(n_sentences=979, n_batches=32, n_buckets=32)
test:  Dataset(n_sentences=979, n_batches=32, n_buckets=32)

INFO:supar:Epoch 1 / 50:
  0%|                  | 0/41 00:00<?, ?it/s/usr/local/lib/python3.12/dist-packages/supar/utils/fn.py:155: UserWarning: Using a non-tuple sequence for

---
## 7 — A2a: BiLSTM + BERTurk, no POS

Identical to A1b — copies checkpoint under the A2a label.

In [10]:
# A2a: BiLSTM + BERTurk, no POS — identical to A1b, just a different label
import shutil
A2A_PATH = os.path.join(ABLATION_DIR, "a2a_lstm_no_pos.pt")
if os.path.exists(A1B_PATH) and not os.path.exists(A2A_PATH):
    shutil.copy2(A1B_PATH, A2A_PATH)
    print("Copied a1b -> a2a")
elif os.path.exists(A2A_PATH):
    print("A2a already exists")
else:
    print("A1b not trained yet — run section 5 first")
parser_a2a = parser_a1b
print("A2a = A1b (same model, no POS features)")


Copied a1b -> a2a
A2a = A1b (same model, no POS features)


---
## 8 — A2b: BiLSTM + BERTurk + POS tags

Expected runtime: ~2–4 hours on A100. Best ablation configuration.

In [11]:
# A2b: BiLSTM + BERTurk + POS tags — best ablation configuration
# Expected runtime: ~2-4 hours on A100
parser_a2b = train_lstm_mode("a2b_lstm_with_pos", feat=["bert", "tag"])


INFO:supar.parsers.dep:Building the fields
/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(



 a2b_lstm_with_pos | encoder=lstm  feat=['bert', 'tag']
 LR=0.002  EPOCHS=50  PATIENCE=15


INFO:supar.parsers.dep:CoNLL(
 (words): Field(pad=<pad>, unk=<unk>, bos=<bos>, lower=True)
 (texts): RawField()
 (bert): SubwordField(pad=[PAD], unk=[UNK], bos=[CLS])
 (tags): Field(bos=<bos>)
 (arcs): Field(bos=<bos>, use_vocab=False)
 (sibs): ChartField(bos=<bos>, use_vocab=False)
 (rels): Field(bos=<bos>)
)
INFO:supar.parsers.dep:Building the model
INFO:supar.parsers.dep:CRF2oDependencyModel(
  (word_embed): Embedding(9081, 100)
  (tag_embed): Embedding(17, 100)
  (bert_embed): TransformerEmbedding(dbmdz/bert-base-turkish-uncased, n_layers=4, n_out=100, stride=256, pooling=mean, pad_index=0)
  (embed_dropout): IndependentDropout(p=0.33)
  (encoder): VariationalLSTM(300, 400, num_layers=3, bidirectional=True, dropout=0.33)
  (encoder_dropout): SharedDropout(p=0.33, batch_first=True)
  (arc_mlp_d): MLP(n_in=800, n_out=500, dropout=0.33)
  (arc_mlp_h): MLP(n_in=800, n_out=500, dropout=0.33)
  (rel_mlp_d): MLP(n_in=800, n_out=100, dropout=0.33)
  (rel_mlp_h): MLP(n_in=800, n_out=100, dr

---
## 9 — A3b: XLM-R encoder

Lower LR (5e-4 task, 5e-6 encoder) and smaller batch (2000) vs BERTurk.
Expected runtime: ~3–5 hours on A100.

In [12]:
# A3b: XLM-R encoder — lower LR needed vs BERTurk
# Expected runtime: ~3-5 hours on A100 (larger model, smaller batch)
parser_a3b = train_bert_mode(
    "a3b_xlmr",
    encoder_name = XLMR_NAME,
    lr           = XLMR_LR,
    encoder_lr   = XLMR_ENCODER_LR,
    max_epochs   = XLMR_MAX_EPOCHS,
    patience     = XLMR_PATIENCE,
    batch_size   = XLMR_BATCH,
)


INFO:supar.parsers.dep:Building the fields



 a3b_xlmr | encoder=xlm-roberta-base
 LR=0.0005  ENCODER_LR=5e-06  EPOCHS=50  PATIENCE=15


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

INFO:supar.parsers.dep:CoNLL(
 (words): SubwordField(pad=<pad>, unk=<unk>, bos=<s>)
 (texts): RawField()
 (arcs): Field(bos=<bos>, use_vocab=False)
 (sibs): ChartField(bos=<bos>, use_vocab=False)
 (rels): Field(bos=<bos>)
)
INFO:supar.parsers.dep:Building the model


model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

INFO:supar.parsers.dep:CRF2oDependencyModel(
  (encoder): TransformerEmbedding(xlm-roberta-base, n_layers=4, n_out=768, stride=256, pooling=mean, pad_index=1, requires_grad=True)
  (encoder_dropout): Dropout(p=0.33, inplace=False)
  (arc_mlp_d): MLP(n_in=768, n_out=500, dropout=0.33)
  (arc_mlp_h): MLP(n_in=768, n_out=500, dropout=0.33)
  (rel_mlp_d): MLP(n_in=768, n_out=100, dropout=0.33)
  (rel_mlp_h): MLP(n_in=768, n_out=100, dropout=0.33)
  (arc_attn): Biaffine(n_in=500, bias_x=True)
  (rel_attn): Biaffine(n_in=100, n_out=48, bias_x=True, bias_y=True)
  (criterion): CrossEntropyLoss()
  (sib_mlp_s): MLP(n_in=768, n_out=100, dropout=0.33)
  (sib_mlp_d): MLP(n_in=768, n_out=100, dropout=0.33)
  (sib_mlp_h): MLP(n_in=768, n_out=100, dropout=0.33)
  (sib_attn): Triaffine(n_in=100, bias_x=True, bias_y=True)
)

INFO:supar:Loading the data
  0%|                  | 0/7803 00:00<?, ?it/s/usr/local/lib/python3.12/dist-packages/supar/utils/fn.py:155: UserWarning: Using a non-tuple sequence fo

---
## 10 — Evaluate All Systems

`force_rerun=True` ensures fresh predictions after training.
Evaluation plan:
- A1a / A3a: BOUN + PUD
- A1b / A2a: BOUN only
- A2b / A3b: BOUN + PUD

In [13]:
import json

results = []

# Evaluation plan:
#   A1a / A3a: BOUN + PUD  (same parser — baseline_boun.pt)
#   A1b / A2a: BOUN only   (same model)
#   A2b:       BOUN + PUD
#   A3b:       BOUN + PUD
eval_plan = [
    (parser_a1a, "a1a", [("BOUN", BOUN["test"]), ("PUD", PUD["test"])]),
    (parser_a1b, "a1b", [("BOUN", BOUN["test"])]),
    (parser_a2a, "a2a", [("BOUN", BOUN["test"])]),
    (parser_a2b, "a2b", [("BOUN", BOUN["test"]), ("PUD", PUD["test"])]),
    (parser_a3a, "a3a", [("BOUN", BOUN["test"]), ("PUD", PUD["test"])]),
    (parser_a3b, "a3b", [("BOUN", BOUN["test"]), ("PUD", PUD["test"])]),
]

for parser_obj, sys_name, test_sets in eval_plan:
    for test_name, test_path in test_sets:
        r = evaluate_and_save(parser_obj, sys_name, test_path,
                              test_name, force_rerun=True)
        results.append(r)
    if sys_name in ("a1b", "a2b", "a3b"):
        torch.cuda.empty_cache()

results_path = os.path.join(RESULTS_DIR, "ablation_results.json")
with open(results_path, "w") as f:
    json.dump(results, f, indent=2)
print(f"\nResults saved: {results_path}")


  [a1a] → BOUN


/usr/local/lib/python3.12/dist-packages/supar/utils/fn.py:155: UserWarning: Using a non-tuple sequence for multidimensional indexing is deprecated and will be changed in pytorch 2.9; use x[tuple(seq)] instead of x[seq]. In pytorch 2.9 this will be interpreted as tensor index, x[torch.tensor(seq)], which will result either in an error or a different result (Triggered internally at /pytorch/torch/csrc/autograd/python_variable_indexing.cpp:347.)
  out_tensor[i][[slice(-i, None) if padding_side == 'left' else slice(0, i) for i in tensor.size()]] = tensor


    Saved: a1a_BOUN.conllu (979 sentences)
    UAS 77.26  LAS 61.96  npy: a1a_BOUN_sent_las.npy
  [a1a] → PUD
    Saved: a1a_PUD.conllu (1000 sentences)
    UAS 76.26  LAS 60.41  npy: a1a_PUD_sent_las.npy
  [a1b] → BOUN


/usr/local/lib/python3.12/dist-packages/supar/modules/lstm.py:152: FutureWarning: `apply_permutation` is deprecated, please use `tensor.index_select(dim, permutation)` instead
  h = apply_permutation(hx[0], permutation)
/usr/local/lib/python3.12/dist-packages/supar/modules/lstm.py:153: FutureWarning: `apply_permutation` is deprecated, please use `tensor.index_select(dim, permutation)` instead
  c = apply_permutation(hx[1], permutation)


    Saved: a1b_BOUN.conllu (979 sentences)
    UAS 82.06  LAS 72.84  npy: a1b_BOUN_sent_las.npy
  [a2a] → BOUN
    Saved: a2a_BOUN.conllu (979 sentences)
    UAS 82.06  LAS 72.84  npy: a2a_BOUN_sent_las.npy
  [a2b] → BOUN
    Saved: a2b_BOUN.conllu (979 sentences)
    UAS 82.62  LAS 74.73  npy: a2b_BOUN_sent_las.npy
  [a2b] → PUD
    Saved: a2b_PUD.conllu (1000 sentences)
    UAS 83.19  LAS 69.41  npy: a2b_PUD_sent_las.npy
  [a3a] → BOUN
    Saved: a3a_BOUN.conllu (979 sentences)
    UAS 77.26  LAS 61.96  npy: a3a_BOUN_sent_las.npy
  [a3a] → PUD
    Saved: a3a_PUD.conllu (1000 sentences)
    UAS 76.26  LAS 60.41  npy: a3a_PUD_sent_las.npy
  [a3b] → BOUN
    Saved: a3b_BOUN.conllu (979 sentences)
    UAS 76.05  LAS 49.39  npy: a3b_BOUN_sent_las.npy
  [a3b] → PUD
    Saved: a3b_PUD.conllu (1000 sentences)
    UAS 74.52  LAS 48.90  npy: a3b_PUD_sent_las.npy

Results saved: /content/drive/MyDrive/58T - Application Project/results/ablation_results.json


---
## 11 — Results Table

In [14]:
LABELS = {
    "a1a": "A1a: BERTurk, no BiLSTM  (baseline)",
    "a1b": "A1b: BiLSTM + BERTurk",
    "a2a": "A2a: BiLSTM + BERTurk, no POS",
    "a2b": "A2b: BiLSTM + BERTurk + POS",
    "a3a": "A3a: BERTurk encoder",
    "a3b": "A3b: XLM-R encoder",
}

baseline = next((r for r in results if r["system"] == "a1a" and r["test"] == "BOUN"), None)
baseline_las = baseline["LAS"] if baseline else None

print(f"{'System':<42} {'Test':<6} {'UAS':>7} {'LAS':>7} {'ΔLAS':>7}")
print("-" * 75)
for r in results:
    label = LABELS.get(r["system"], r["system"])
    delta = ""
    if baseline_las is not None and r["test"] == "BOUN":
        d = r["LAS"] - baseline_las
        delta = f"{d:+.2f}"
    print(f"  {label:<40} {r['test']:<6} {r['UAS']:>7.2f} {r['LAS']:>7.2f} {delta:>7}")


System                                     Test       UAS     LAS    ΔLAS
---------------------------------------------------------------------------
  A1a: BERTurk, no BiLSTM  (baseline)      BOUN     77.26   61.96   +0.00
  A1a: BERTurk, no BiLSTM  (baseline)      PUD      76.26   60.41        
  A1b: BiLSTM + BERTurk                    BOUN     82.06   72.84  +10.88
  A2a: BiLSTM + BERTurk, no POS            BOUN     82.06   72.84  +10.88
  A2b: BiLSTM + BERTurk + POS              BOUN     82.62   74.73  +12.77
  A2b: BiLSTM + BERTurk + POS              PUD      83.19   69.41        
  A3a: BERTurk encoder                     BOUN     77.26   61.96   +0.00
  A3a: BERTurk encoder                     PUD      76.26   60.41        
  A3b: XLM-R encoder                       BOUN     76.05   49.39  -12.57
  A3b: XLM-R encoder                       PUD      74.52   48.90        


---
## 12 — Per-Relation LAS Breakdown (A2b on BOUN)

In [15]:
import json

pred_a2b_path = os.path.join(PRED_DIR, "a2b_BOUN.conllu")

if not os.path.exists(pred_a2b_path):
    print("A2b BOUN prediction file not found — run section 8 (Evaluate) first.")
else:
    gold_boun = load_gold(BOUN["test"])
    with open(pred_a2b_path, encoding="utf-8") as f:
        pred_a2b_sents = conllu_lib.parse(f.read())

    breakdown = per_relation_las(gold_boun, pred_a2b_sents)
    br_path = os.path.join(RESULTS_DIR, "a2b_BOUN_per_rel.json")
    with open(br_path, "w") as f:
        json.dump(breakdown, f, indent=2)
    print(f"Per-relation breakdown saved: {br_path}")
    print()
    print(f"{'Relation':<15} {'Count':>7} {'LAS':>7}")
    print("-" * 32)
    for rel, vals in list(breakdown.items())[:20]:
        print(f"  {rel:<13} {vals['total']:>7} {vals['LAS']:>7.2f}")


Per-relation breakdown saved: /content/drive/MyDrive/58T - Application Project/results/a2b_BOUN_per_rel.json

Relation          Count     LAS
--------------------------------
  punct            2026   78.43
  nmod:poss        1053   84.62
  obl               987   75.28
  root              979   88.97
  nsubj             829   76.36
  amod              766   71.80
  obj               728   72.25
  conj              679   63.62
  det               524   91.98
  advmod            436   70.87
  nmod              366   43.44
  acl               351   81.48
  advcl             317   64.35
  case              294   85.37
  cc                258   72.87
  compound          217   42.86
  cop               192   83.85
  flat              181   77.90
  ccomp             173   82.08
  nummod            163   74.85


---
## 13 — Verify Saved Files

In [16]:
expected = [
    (ABLATION_DIR, "a1b_lstm_bert_feat.pt"),
    (ABLATION_DIR, "a2b_lstm_with_pos.pt"),
    (ABLATION_DIR, "a3b_xlmr.pt"),
    (PRED_DIR,     "a1a_BOUN.conllu"),
    (PRED_DIR,     "a1a_BOUN_sent_las.npy"),
    (PRED_DIR,     "a1a_PUD.conllu"),
    (PRED_DIR,     "a1a_PUD_sent_las.npy"),
    (PRED_DIR,     "a1b_BOUN.conllu"),
    (PRED_DIR,     "a1b_BOUN_sent_las.npy"),
    (PRED_DIR,     "a2a_BOUN.conllu"),
    (PRED_DIR,     "a2a_BOUN_sent_las.npy"),
    (PRED_DIR,     "a2b_BOUN.conllu"),
    (PRED_DIR,     "a2b_BOUN_sent_las.npy"),
    (PRED_DIR,     "a2b_PUD.conllu"),
    (PRED_DIR,     "a2b_PUD_sent_las.npy"),
    (PRED_DIR,     "a3a_BOUN.conllu"),
    (PRED_DIR,     "a3a_BOUN_sent_las.npy"),
    (PRED_DIR,     "a3a_PUD.conllu"),
    (PRED_DIR,     "a3a_PUD_sent_las.npy"),
    (PRED_DIR,     "a3b_BOUN.conllu"),
    (PRED_DIR,     "a3b_BOUN_sent_las.npy"),
    (PRED_DIR,     "a3b_PUD.conllu"),
    (PRED_DIR,     "a3b_PUD_sent_las.npy"),
    (RESULTS_DIR,  "ablation_results.json"),
    (RESULTS_DIR,  "a2b_BOUN_per_rel.json"),
]

print(f"{'Status':<8} {'Size':>12}  {'Dir':<14} File")
print("-" * 85)
all_ok = True
for directory, filename in expected:
    path   = os.path.join(directory, filename)
    exists = os.path.exists(path)
    size   = f"{os.path.getsize(path):,} B" if exists else "-"
    status = "OK" if exists else "MISSING"
    folder = os.path.basename(directory)
    if not exists: all_ok = False
    print(f"  {status:<6} {size:>12}  {folder:<14} {filename}")

print()
if all_ok:
    print("All files present. 03_ablation complete.")
    print("Next: 04_adapters.ipynb")
else:
    print("Some files missing — rerun the relevant sections above.")


Status           Size  Dir            File
-------------------------------------------------------------------------------------
  OK     498,253,077 B  ablation       a1b_lstm_bert_feat.pt
  OK     499,540,451 B  ablation       a2b_lstm_with_pos.pt
  OK     1,132,468,458 B  ablation       a3b_xlmr.pt
  OK        965,615 B  predictions    a1a_BOUN.conllu
  OK          1,107 B  predictions    a1a_BOUN_sent_las.npy
  OK      1,385,714 B  predictions    a1a_PUD.conllu
  OK          1,128 B  predictions    a1a_PUD_sent_las.npy
  OK        965,801 B  predictions    a1b_BOUN.conllu
  OK          1,107 B  predictions    a1b_BOUN_sent_las.npy
  OK        965,801 B  predictions    a2a_BOUN.conllu
  OK          1,107 B  predictions    a2a_BOUN_sent_las.npy
  OK        965,873 B  predictions    a2b_BOUN.conllu
  OK          1,107 B  predictions    a2b_BOUN_sent_las.npy
  OK      1,384,839 B  predictions    a2b_PUD.conllu
  OK          1,128 B  predictions    a2b_PUD_sent_las.npy
  OK        965,6